In [ ]:
# Import libraries.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load data for modeling
df = pd.read_parquet("/Users/ben/Projects/Instacart/data/final.parquet")

In [3]:
# inpsect data
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [4]:
# Create target variable; calculate days until each customer's
# next order.
# Match those intervals to the modeling df by order_id.
df["next_order_within_7_days"] = df["order_id"].map(
    pd.read_csv(
        "/Users/ben/Projects/Instacart/data/raw/orders.csv",
        usecols=["order_id", "user_id", "order_number", "days_since_prior_order"]
    )
    .sort_values(["user_id", "order_number"])
    .set_index("order_id")
    .groupby("user_id")["days_since_prior_order"]
    # The next order's prior-order gap is the current order's
    # wait until the next order.
    .shift(-1)
)

# Create target: True if next order occurs within 7 days,
# otherwise false.
# Preserve missing intervals as unknown instead of labeling false.
df["next_order_within_7_days"] = (
    df["next_order_within_7_days"]
    .le(7)
    .where(df["next_order_within_7_days"].notna())
    .astype("boolean")
)

df

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department,next_order_within_7_days
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs,False
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce,False
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry,False
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry,False
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33819101,3421063,14233,3,1,169679,train,30,0,10,4.0,Natural Artesian Water,115,7,water seltzer sparkling water,beverages,<NA>
33819102,3421063,35548,4,1,169679,train,30,0,10,4.0,Twice Baked Potatoes,13,20,prepared meals,deli,<NA>
33819103,3421070,35951,1,1,139822,train,15,6,10,8.0,Organic Unsweetened Almond Milk,91,16,soy lactosefree,dairy eggs,<NA>
33819104,3421070,16953,2,1,139822,train,15,6,10,8.0,Creamy Peanut Butter,88,13,spreads,pantry,<NA>
